# TabTune · Real-World Cash-Flow Forecasting
### The US Daily Treasury Statement: 490,000 rows of daily government cash flows

**Library:** TabTune open source (`tabtune.TimeSeries`) · **Runtime:** Colab Pro+ with a GPU (A100 or L4)

**The data.** Every business day the US Treasury publishes the deposits into and withdrawals from its operating cash account, line by line: tax receipts, Social Security, Medicare, defence vendors, interest, tax refunds and about 150 other categories. It is public, updated daily, and goes back to 2005. It is the closest public analogue to a bank treasury's cash ledger: large, daily, calendar-driven, and messy.

**Why it is hard.** The raw feed was never designed for modelling:
* categories are **free-text labels that were renamed four times** (2012, 2019–22, 2023) — the same flow appears under different names in different years;
* totals, sub-totals, debt financing and internal transfers sit in the same table as operating flows;
* amounts can be negative (reversals), and there are no rows on federal holidays;
* the report layout itself changed (three different account structures).

| Part | What you will see |
|---|---|
| 1 | Download from the Treasury API, raw profile |
| 2 | **Data preparation, visualised**: structural rows, label harmonisation across eras, continuity checks, calendar gaps |
| 3 | **Feature engineering, visualised**: scaling, seasonality, autocorrelation, decomposition, payment-calendar and tax-calendar features, feature importance |
| 4 | Model race on rolling backtests |
| 5 | 4-week (20 business day) forecast with calendar covariates and calibrated bands |
| 6 | Treasury dashboard: net operating cash position |
| 7 | Ensembles and series similarity |
| 8 | Anomaly replay: would the monitor have caught the COVID fiscal shock? |

Source: US Department of the Treasury, Bureau of the Fiscal Service, [FiscalData — Daily Treasury Statement](https://fiscaldata.treasury.gov/datasets/daily-treasury-statement/) (public domain). Amounts are in USD millions.

**Run:** *Runtime → Change runtime type → GPU*, then *Run all*. The download takes about two minutes and is cached.

In [ ]:
# ⚙️ Install TabTune (open source) on Colab — about 2–3 minutes
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "tabtune[colab] @ git+https://github.com/Lexsi-Labs/TabTune.git", "plotly"], check=True)
print("Colab:", IN_COLAB)

In [ ]:
# 🔧 Imports and shared settings
import logging, time, warnings, re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from tabtune.logger import setup_logger
from tabtune.TimeSeries import (TimeSeriesPipeline, TimeSeriesSchema, TimeSeriesLeaderboard,
                                TimeSeriesEnsemble, split_horizon)

warnings.filterwarnings("ignore")
setup_logger(level=logging.WARNING)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
pd.set_option("display.max_columns", 40)

# FAST=1 swaps foundation models for weight-free models (smoke test without GPU or downloads)
FAST = os.environ.get("TABTUNE_DEMO_FAST") == "1"
QUANTILES = [0.05, 0.1, 0.5, 0.9, 0.95]
FOUNDATION_MODELS = [] if FAST else ["ChronosBolt", "Chronos2"]     # Apache-2.0 weights: commercial-safe
BASELINES = ["SeasonalNaive", "WindowAverage", "TabularTS-GBM"]
PRIMARY = "TabularTS-GBM" if FAST else "Chronos2"                   # reads covariates
DETECTOR = "SeasonalNaive" if FAST else "ChronosBolt"
EMBEDDER = None if FAST else "ChronosBolt"
RUN_FINETUNE = False                                                # optional LoRA section
GBM_PARAMS = {"max_iter": 60}                                       # keeps the tree baseline quick
PRIMARY_PARAMS = GBM_PARAMS if PRIMARY == "TabularTS-GBM" else {}

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"
BLUE, ORANGE, TEAL, GREY, DARK, RED, GOLD = "#2F5BEA", "#B4541A", "#1B998B", "#9AA0A8", "#16181D", "#D7263D", "#E8C547"
DOW = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
MON = ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
print(f"Device: {DEVICE} · primary model: {PRIMARY} · detector: {DETECTOR} · fast mode: {FAST}")

### Model support in TabTune time series

The dictionary below is built from the library's own registry, so it always matches the installed version.

In [ ]:
# 📚 Time-series models available in TabTune today (read live from the library's registry)
from tabtune.registry import list_time_series_models

def model_kind(spec):
    if spec.name.startswith("TabularTS") or spec.name == "TabPFN-TS":
        return "Tabular model as forecaster"
    return "Foundation model" if spec.default_checkpoint else "Statistical baseline"

TS_MODELS = {
    spec.name: {
        "kind": model_kind(spec),
        "family": spec.family,
        "tasks": sorted(spec.tasks),
        "strategies": sorted(spec.strategies),          # inference = zero-shot, finetune, peft = LoRA
        "default_checkpoint": spec.default_checkpoint or "none (no weights)",
        "max_context": spec.max_context,
        "multivariate": spec.supports_multivariate,
        "covariates": spec.supports_covariates,
        "licence": spec.license.name,
        "commercial_use": {True: "yes", False: "no", None: "unverified"}[spec.license.commercial_use_ok],
    }
    for spec in list_time_series_models()
}

catalogue_ts = pd.DataFrame(TS_MODELS).T.rename_axis("model")
print(f"{len(TS_MODELS)} models registered:", catalogue_ts["kind"].value_counts().to_dict())
for kind in ("Foundation model", "Tabular model as forecaster", "Statistical baseline"):
    part = catalogue_ts[catalogue_ts.kind == kind].drop(columns="kind")
    part = part.assign(tasks=part.tasks.map(", ".join), strategies=part.strategies.map(", ".join))
    display(part.style.set_caption(f"{kind} · {len(part)} models"))
TS_MODELS["Chronos2"]          # one entry of the dictionary

In [ ]:
# 🔧 Notebook settings
HORIZON, SEASON, MONTH_LAG, SCALE_WINDOW = 20, 5, 21, 250      # business days: 4-week horizon, 5-day week
UNIT = "USD mn"
MODEL_START = "2016-10-03"                                      # start of fiscal year 2017
GROUP_COLORS = {"Inflow": TEAL, "Outflow": ORANGE}
DTS_CACHE = os.environ.get("DTS_CACHE", "/content/dts_raw.parquet")

## 1 · Download and first look

In [ ]:
# 📥 Daily Treasury Statement, table II (deposits and withdrawals of operating cash), full history
import requests

API = "https://api.fiscaldata.treasury.gov/services/api/fiscal_service/v1/accounting/dts/deposits_withdrawals_operating_cash"
FIELDS = "record_date,account_type,transaction_type,transaction_catg,transaction_catg_desc,transaction_today_amt"

if os.path.exists(DTS_CACHE):
    raw = pd.read_parquet(DTS_CACHE)
    print("loaded cache:", DTS_CACHE)
else:
    rows, page, t0 = [], 1, time.time()
    while True:
        r = requests.get(API, params={"fields": FIELDS, "sort": "record_date", "page[size]": 10000, "page[number]": page}, timeout=120)
        r.raise_for_status()
        j = r.json()
        rows += j["data"]
        if page >= j["meta"]["total-pages"]:
            break
        page += 1
    raw = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(DTS_CACHE) or ".", exist_ok=True)
    raw.to_parquet(DTS_CACHE)
    print(f"downloaded {page} pages in {time.time() - t0:.0f}s")

raw["date"] = pd.to_datetime(raw["record_date"])
raw["amount"] = pd.to_numeric(raw["transaction_today_amt"], errors="coerce")
raw["category"] = raw["transaction_catg"].str.strip()
print(f"{len(raw):,} rows · {raw.date.nunique():,} statement days · {raw.date.min():%d %b %Y} → {raw.date.max():%d %b %Y} · "
      f"{raw.category.nunique()} distinct category labels")
display(raw[["record_date", "account_type", "transaction_type", "transaction_catg", "transaction_today_amt"]].sample(6, random_state=1))

In [ ]:
# 🔎 Raw profile: the feed changes shape over time
per_day = raw.groupby("date").size()
labels_per_year = raw.groupby(raw.date.dt.year)["category"].nunique()
acct = raw.groupby("account_type")["date"].agg(["min", "max"]).sort_values("min")
neg = raw[raw.amount < 0].groupby(raw.date.dt.year).size().reindex(labels_per_year.index, fill_value=0)

fig = make_subplots(rows=2, cols=2, vertical_spacing=0.16,
                    subplot_titles=("Rows per statement day", "Distinct category labels in use, by year",
                                    "Account structure over time", "Negative amounts (reversals) per year"))
fig.add_trace(go.Scattergl(x=per_day.index, y=per_day.values, mode="markers", marker=dict(size=3, color=BLUE), showlegend=False), 1, 1)
fig.add_trace(go.Bar(x=labels_per_year.index, y=labels_per_year.values, marker_color=ORANGE, showlegend=False), 1, 2)
for i, (name, r) in enumerate(acct.iterrows()):
    fig.add_trace(go.Scatter(x=[r["min"], r["max"]], y=[name[:34], name[:34]], mode="lines", line=dict(width=12, color=[BLUE, ORANGE, TEAL, GREY, RED, GOLD, DARK][i % 7]),
                             showlegend=False), 2, 1)
fig.add_trace(go.Bar(x=neg.index, y=neg.values, marker_color=RED, showlegend=False), 2, 2)
fig.update_layout(template="plotly_white", height=720, title="The raw feed · three layouts, shifting label sets, reversals")
fig.show()

## 2 · Data preparation, visualised

### 2a · Separate operating flows from structural rows
The table mixes four kinds of row. Only **operating** flows are a forecasting target; debt issuance and redemption are financing decisions, transfers move cash between the Treasury's own accounts, and totals would double count.

In [ ]:
# 🧱 Classify every row: operating, financing, internal transfer, or total
def row_kind(cat, acct):
    c = cat.lower()
    if c in ("null", "") or "total" in c or "total" in acct.lower():
        return "total / sub-total"
    if c.startswith("public debt"):
        return "debt financing"
    if c.startswith("transfers ") or "(table v)" in c:
        return "internal transfer"
    return "operating"

kinds = {(c, a): row_kind(c, a) for c, a in raw[["category", "account_type"]].drop_duplicates().itertuples(index=False)}
raw["kind"] = [kinds[k] for k in zip(raw.category, raw.account_type)]
by_kind = raw.groupby([raw.date.dt.year, "kind"])["amount"].sum().unstack(fill_value=0) / 1e6
KIND_COLORS = {"operating": BLUE, "debt financing": GREY, "internal transfer": GOLD, "total / sub-total": RED}
fig = make_subplots(rows=1, cols=2, column_widths=[0.65, 0.35], subplot_titles=("Gross amount by row type and year (USD trillion)", "Rows by type"))
for k in KIND_COLORS:
    fig.add_trace(go.Bar(x=by_kind.index, y=by_kind[k], name=k, marker_color=KIND_COLORS[k]), 1, 1)
cnt = raw.kind.value_counts()
fig.add_trace(go.Bar(x=cnt.index, y=cnt.values, marker_color=[KIND_COLORS[k] for k in cnt.index], text=cnt.values,
                     texttemplate="%{text:,}", showlegend=False), 1, 2)
fig.update_layout(template="plotly_white", height=460, barmode="stack", legend=dict(orientation="h", y=-0.15),
                  title="Most of the money in the table is not an operating flow")
fig.show()
ops = raw[raw.kind == "operating"].copy()
print(f"operating rows kept: {len(ops):,} of {len(raw):,}")

### 2b · Harmonise category labels across eras
The same economic flow carries different labels in different years (for example `Social Security Benefits (EFT)` became `SSA - Benefits Payments` in 2021, and Medicare went from one line to two to three). We map the raw labels onto **16 canonical cash-flow lines** with explicit rules, and collect everything else into two residual lines so the net position stays complete.

In [ ]:
# 🏷️ Canonical lines: (direction, label patterns). Patterns are regular expressions on the raw label.
LINES = {
    "Federal tax deposits":           ("Deposits",    [r"^(cash )?ftd's received", r"^taxes - (withheld|corporate|miscellaneous excise|federal unemployment|railroad)"]),
    "Individual taxes, not withheld": ("Deposits",    [r"^individual income and employment taxes, not withheld", r"^taxes - non withheld"]),
    "Customs and excise":             ("Deposits",    [r"customs"]),
    "Postal Service receipts":        ("Deposits",    [r"^postal service$", r"^united states postal service"]),
    "Education receipts":             ("Deposits",    [r"^education department programs", r"^dept of education"]),
    "Social Security benefits":       ("Withdrawals", [r"^social security benefits \(eft\)", r"^ssa - benefits payments"]),
    "Medicare":                       ("Withdrawals", [r"^medicare$", r"^medicare and other cms", r"^medicare advantage", r"^hhs - federal (supple med|hospital) insr", r"^hhs - medicare prescription"]),
    "Medicaid":                       ("Withdrawals", [r"^medicaid$", r"^hhs - grants to states for medicaid"]),
    "Defense payments":               ("Withdrawals", [r"^defense vendor payments", r"^dept of defense \(dod\) - misc"]),
    "Federal salaries":               ("Withdrawals", [r"^federal salaries"]),
    "Interest on Treasury securities": ("Withdrawals", [r"^interest on treasury securities"]),
    "Education payments":             ("Withdrawals", [r"^education department programs", r"^dept of education"]),
    "Unemployment benefits":          ("Withdrawals", [r"^unemployment insurance benefits", r"^dol - unemployment benefits"]),
    "SNAP (food assistance)":         ("Withdrawals", [r"nutrition assist"]),
    "Veterans programs":              ("Withdrawals", [r"^veterans affairs programs", r"^dept of veterans affairs", r"^va - benefits"]),
    "Individual tax refunds":         ("Withdrawals", [r"^irs tax refunds individual", r"^taxes - individual tax refunds"]),
}

def canonical(cat, direction):
    c = cat.lower()
    for line, (d, patterns) in LINES.items():
        if d == direction and any(re.search(p, c) for p in patterns):
            return line
    return "Other deposits" if direction == "Deposits" else "Other withdrawals"

pairs = ops[["category", "transaction_type"]].drop_duplicates()
lookup = {(c, t): canonical(c, t) for c, t in pairs.itertuples(index=False)}
ops["line"] = [lookup[k] for k in zip(ops.category, ops.transaction_type)]

mapped = (ops[~ops.line.str.startswith("Other")].groupby(["line", "category"])
          .agg(first=("date", "min"), last=("date", "max"), total=("amount", "sum")).reset_index())
mapped = mapped[mapped["last"] >= "2012-01-01"].sort_values(["line", "first"])
fig = px.timeline(mapped, x_start="first", x_end="last", y="category", color="line", height=30 * len(mapped) + 160,
                  title=f"{len(mapped)} raw labels stitched into {mapped.line.nunique()} canonical lines · each bar is one label's lifetime")
fig.update_yaxes(categoryorder="array", categoryarray=mapped.category.tolist()[::-1], title="")
fig.update_layout(template="plotly_white", legend=dict(orientation="h", y=-0.06, title=""))
fig.show()

share = ops[ops.date >= "2012-01-01"].assign(named=lambda d: ~d.line.str.startswith("Other")).groupby([ops.date.dt.year, "named"])["amount"].sum().unstack()
print("share of operating volume covered by the 16 named lines, by year (%):")
display((share[True] / share.sum(axis=1) * 100).round(1).to_frame("named lines %").T)

In [ ]:
# ✅ Continuity check: does a line keep its level across each relabelling?
daily = ops.groupby(["line", "date"])["amount"].sum().rename("value").reset_index()
breaks = []
for line, g in mapped.groupby("line"):
    for d0 in sorted(g["first"].unique())[1:]:
        s = daily[daily.line == line].set_index("date")["value"]
        before, after = s[d0 - pd.Timedelta(days=365):d0 - pd.Timedelta(days=1)].sum(), s[d0:d0 + pd.Timedelta(days=364)].sum()
        breaks.append({"line": line, "relabelled on": d0.date(), "12 months before": before, "12 months after": after,
                       "change_%": (after / before - 1) * 100 if before else np.nan})
breaks = pd.DataFrame(breaks)
display(breaks.style.format({"12 months before": "{:,.0f}", "12 months after": "{:,.0f}", "change_%": "{:+.1f}%"})
        .background_gradient(subset=["change_%"], cmap="RdBu_r", vmin=-40, vmax=40)
        .set_caption("Annual volume either side of each relabelling (USD mn). Small changes mean the stitch is clean; "
                     "large ones are real events (for example COVID in 2020) or a scope change worth a note."))

show = "Medicare"
era = ops[ops.line == show].groupby(["category", pd.Grouper(key="date", freq="MS")])["amount"].sum().reset_index()
stitched = daily[daily.line == show].set_index("date")["value"].resample("MS").sum()
fig = go.Figure()
for cat, g in era.groupby("category"):
    fig.add_trace(go.Bar(x=g.date, y=g.amount, name=cat))
fig.add_trace(go.Scatter(x=stitched.index, y=stitched.values, name="stitched line", line=dict(color=DARK, width=2)))
fig.update_layout(template="plotly_white", height=460, barmode="stack", legend=dict(orientation="h", y=-0.2),
                  title=f"{show} · six raw labels over three eras become one continuous monthly series (USD mn)")
fig.show()

### 2c · Regular business-day grid, holidays and reversals
Models need one value per series per step. There is no statement on federal holidays, so those days are added with a zero flow and flagged. Reversals (negative amounts) are kept: they are real cash movements.

In [ ]:
# 📅 Business-day grid, holiday handling, final panel
from pandas.tseries.holiday import USFederalHolidayCalendar

daily = daily[daily.date >= MODEL_START]
bdays = pd.bdate_range(daily.date.min(), daily.date.max())
missing = bdays.difference(pd.DatetimeIndex(daily.date.unique()))
fed = USFederalHolidayCalendar().holidays(bdays.min(), bdays.max())
print(f"{len(bdays):,} business days · {len(missing)} without a statement · {missing.isin(fed).sum()} of those are federal holidays")
other = missing[~missing.isin(fed)]
if len(other):
    print("non-holiday gaps (one-off closures):", [d.strftime("%Y-%m-%d") for d in other[:12]])

lines = sorted(daily.line.unique())
grid = pd.MultiIndex.from_product([lines, bdays], names=["item_id", "timestamp"])
panel = (daily.rename(columns={"line": "item_id", "date": "timestamp"}).set_index(["item_id", "timestamp"])
         .reindex(grid, fill_value=0.0).reset_index())
direction = {**{k: v[0] for k, v in LINES.items()}, "Other deposits": "Deposits", "Other withdrawals": "Withdrawals"}
meta = pd.DataFrame({"item_id": lines})
meta["group"] = meta.item_id.map(lambda l: "Inflow" if direction[l] == "Deposits" else "Outflow")
meta = meta.sort_values(["group", "item_id"]).reset_index(drop=True)

yr = panel.assign(group=panel.item_id.map(meta.set_index("item_id")["group"]))
tot = yr.groupby("item_id")["value"].sum().sort_values()
neg_days = (panel.value < 0).groupby(panel.item_id).sum().reindex(tot.index)
zero_days = (panel.value == 0).groupby(panel.item_id).mean().mul(100).reindex(tot.index)
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, subplot_titles=("Total since FY2017 (USD trillion)", "Days with zero flow (%)", "Days with a net reversal"))
fig.add_trace(go.Bar(y=tot.index, x=tot.values / 1e6, orientation="h", showlegend=False,
                     marker_color=[GROUP_COLORS[meta.set_index("item_id").group[i]] for i in tot.index]), 1, 1)
fig.add_trace(go.Bar(y=tot.index, x=zero_days.values, orientation="h", marker_color=GREY, showlegend=False), 1, 2)
fig.add_trace(go.Bar(y=tot.index, x=neg_days.values, orientation="h", marker_color=RED, showlegend=False), 1, 3)
fig.update_layout(template="plotly_white", height=620, title="The modelling panel · teal = inflow, orange = outflow")
fig.show()
print(f"panel: {len(panel):,} rows · {panel.item_id.nunique()} series · {panel.timestamp.min():%d %b %Y} → {panel.timestamp.max():%d %b %Y}")

## 3 · Feature engineering, visualised

Government cash follows rules written in law: Social Security pays on the 3rd and on the second, third and fourth Wednesdays; estimated taxes fall due on fixed dates; salaries follow a two-week cycle; interest lands mid-month and month-end. We turn those rules into **calendar covariates known in advance**.

In [ ]:
# 🗓️ Payment-calendar and tax-calendar features (all known in advance)
def next_bday(d):
    d = pd.Timestamp(d)
    while d.dayofweek >= 5 or d in fed:
        d += pd.Timedelta(days=1)
    return d

ts = pd.DatetimeIndex(bdays)
cal = pd.DataFrame({"timestamp": ts})
cal["day_of_week"] = ts.dayofweek
cal["day_of_month"] = ts.day
month_key = ts.to_period("M")
cal["bday_of_month"] = cal.groupby(month_key).cumcount().to_numpy() + 1
cal["bdays_to_month_end"] = cal.groupby(month_key).cumcount(ascending=False).to_numpy()
cal["is_month_start"] = (cal.bday_of_month == 1).astype(int)
cal["is_month_end"] = (cal.bdays_to_month_end == 0).astype(int)
years = range(ts.year.min(), ts.year.max() + 2)
mids = {next_bday(f"{y}-{m:02d}-15") for y in years for m in range(1, 13)}
thirds = {next_bday(f"{y}-{m:02d}-03") for y in years for m in range(1, 13)}
tax_due = {next_bday(f"{y}-{md_}") for y in years for md_ in ("01-15", "03-15", "04-15", "06-15", "09-15", "12-15")}
cal["is_mid_month"] = ts.isin(mids).astype(int)
cal["is_ss_third"] = ts.isin(thirds).astype(int)                                        # Social Security: the 3rd
wed_no = (ts.day - 1) // 7 + 1
cal["is_ss_wednesday"] = ((ts.dayofweek == 2) & np.isin(wed_no, [2, 3, 4])).astype(int)  # and 2nd–4th Wednesdays
cal["is_tax_due_date"] = ts.isin(tax_due).astype(int)
cal["is_tax_due_week"] = pd.Series(ts.isin(tax_due)).rolling(5, min_periods=1).max().astype(int).to_numpy()
cal["is_refund_season"] = ts.month.isin([2, 3, 4, 5]).astype(int)
cal["is_holiday"] = ts.isin(fed).astype(int)
cal["is_post_holiday"] = cal.is_holiday.shift(1, fill_value=0)
cal["is_fiscal_year_end"] = ((ts.month == 9) & (cal.bdays_to_month_end <= 4)).astype(int)

panel = panel.merge(cal, on="timestamp")
KNOWN = [c for c in cal.columns if c != "timestamp"]
PAST = []
COV_DOC = {"day_of_week": "0 = Monday … 4 = Friday", "day_of_month": "calendar day", "bday_of_month": "business day number within the month",
           "bdays_to_month_end": "business days left in the month", "is_month_start": "first business day of the month",
           "is_month_end": "last business day of the month", "is_mid_month": "the 15th, or the next business day",
           "is_ss_third": "Social Security payment date (the 3rd, or the next business day)",
           "is_ss_wednesday": "Social Security payment Wednesdays (2nd, 3rd, 4th)",
           "is_tax_due_date": "estimated and corporate tax due dates (15 Jan, Mar, Apr, Jun, Sep, Dec)",
           "is_tax_due_week": "the due date and the four business days after it",
           "is_refund_season": "February to May", "is_holiday": "federal holiday (no statement)",
           "is_post_holiday": "first business day after a holiday", "is_fiscal_year_end": "last five business days of September"}
DECOMPOSE_ITEM = "Social Security benefits"

d = panel[panel.timestamp >= panel.timestamp.max() - pd.Timedelta(days=75)]
focus = ["Social Security benefits", "Federal tax deposits", "Interest on Treasury securities"]
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04, row_heights=[0.26, 0.26, 0.26, 0.22],
                    subplot_titles=focus + ["Calendar flags"])
for i, item in enumerate(focus, 1):
    g = d[d.item_id == item]
    fig.add_trace(go.Bar(x=g.timestamp, y=g.value, marker_color=[BLUE, TEAL, ORANGE][i - 1], showlegend=False), i, 1)
flag_rows = ["is_ss_third", "is_ss_wednesday", "is_tax_due_date", "is_mid_month", "is_month_end", "is_holiday"]
z = d[d.item_id == focus[0]].set_index("timestamp")[flag_rows].T
fig.add_trace(go.Heatmap(z=z.values, x=z.columns, y=z.index, colorscale=[[0, "#F7F7F4"], [1, DARK]], showscale=False), 4, 1)
fig.update_layout(template="plotly_white", height=820, title="The last 75 days: flows line up with the engineered calendar flags (USD mn)")
fig.show()

In [ ]:
# 🧰 Feature-engineering helpers used by the plots below
GROUP_OF = meta.set_index("item_id")["group"]


def indexed(panel):
    """Each series divided by its own mean, so series of different size can share an axis."""
    return panel.assign(group=panel.item_id.map(GROUP_OF),
                        idx=panel.value / panel.groupby("item_id")["value"].transform("mean"))


def eta2(v, key):
    """Share of variance explained by the group means of `key` (0 = no pattern, 1 = fully periodic)."""
    m = v.groupby(key).transform("mean")
    tot = ((v - v.mean()) ** 2).sum()
    return float(1 - ((v - m) ** 2).sum() / tot) if tot > 0 else 0.0


def seasonality_strength(panel):
    rows = []
    for item, g in panel.groupby("item_id"):
        v = np.log1p(g["value"].clip(lower=0)).reset_index(drop=True)
        t = g["timestamp"].reset_index(drop=True)
        v = v - v.rolling(SEASON * 26, center=True, min_periods=SEASON * 4).mean().bfill().ffill()   # remove trend
        rows.append({"item_id": item, "day of week": eta2(v, t.dt.dayofweek), "day of month": eta2(v, t.dt.day),
                     "month of year": eta2(v, t.dt.month)})
    return pd.DataFrame(rows).set_index("item_id")


def acf(x, max_lag):
    x = np.asarray(x, float) - np.mean(x)
    d = float(np.dot(x, x)) or 1.0
    return np.array([np.dot(x[:-k], x[k:]) / d for k in range(1, max_lag + 1)])


def decompose(g):
    """Multiplicative split: trend × weekly pattern × day-of-month pattern × remainder."""
    v = g.set_index("timestamp")["value"].astype(float)
    trend = v.rolling(SEASON * 13, center=True, min_periods=SEASON * 4).mean().bfill().ffill().clip(lower=1e-9)
    r1 = v / trend
    weekly = r1.groupby(r1.index.dayofweek).transform("mean").clip(lower=1e-9)
    r2 = r1 / weekly
    monthly = r2.groupby(r2.index.day).transform("mean").clip(lower=1e-9)
    return pd.DataFrame({"observed": v, "trend": trend, "weekly pattern": weekly,
                         "day-of-month pattern": monthly, "remainder": r2 / monthly})


def engineer_features(panel):
    """Lag, rolling and calendar features per series. Every lag is shifted, so nothing looks ahead."""
    out = []
    for item, g in panel.sort_values("timestamp").groupby("item_id"):
        f = pd.DataFrame({"item_id": item, "timestamp": g["timestamp"].values, "y": g["y"].values})
        y = f["y"]
        for l in (1, SEASON, 2 * SEASON, MONTH_LAG):
            f[f"lag_{l}"] = y.shift(l)
        f[f"roll_mean_{SEASON}"] = y.shift(1).rolling(SEASON).mean()
        f[f"roll_mean_{4 * SEASON}"] = y.shift(1).rolling(4 * SEASON).mean()
        f[f"roll_std_{4 * SEASON}"] = y.shift(1).rolling(4 * SEASON).std()
        f["same_weekday_mean_4"] = sum(y.shift(SEASON * k) for k in range(1, 5)) / 4
        f["dow"] = f["timestamp"].dt.dayofweek
        f["dom"] = f["timestamp"].dt.day
        f["month"] = f["timestamp"].dt.month
        f["days_to_month_end"] = f["timestamp"].dt.days_in_month - f["timestamp"].dt.day
        for c in KNOWN:
            f[c] = g[c].values
        for c in PAST:
            f[f"{c}_lag1"] = pd.Series(g[c].values).shift(1).values      # observed only up to yesterday
        out.append(f)
    return pd.concat(out, ignore_index=True).dropna()


def feature_family(name):
    if name.startswith("lag_") or name == "same_weekday_mean_4":
        return "lags"
    if name.startswith("roll_"):
        return "rolling statistics"
    if any(name.startswith(p) for p in PAST):
        return "market drivers"
    return "calendar"


FAMILY_COLORS = {"lags": BLUE, "rolling statistics": TEAL, "calendar": ORANGE, "market drivers": RED}

In [ ]:
# 📏 Model scale: each series as an index of its own recent average
# (errors become comparable between a small and a large series; forecasts are converted back for the dashboards)
panel = panel.sort_values(["item_id", "timestamp"]).reset_index(drop=True)
_in_history = panel.groupby("item_id").cumcount(ascending=False) >= HORIZON      # scale never sees the hold-out
SCALE = (panel[_in_history].groupby("item_id")["value"]
         .apply(lambda s: s.tail(SCALE_WINDOW).abs().mean()).clip(lower=1e-9).rename("scale"))
panel["y"] = panel["value"] / panel["item_id"].map(SCALE)

sample = meta.groupby("group").head(2).item_id
fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Raw level ({UNIT}, log axis)", "Indexed to own recent average"))
for item in sample:
    g = panel[panel.item_id == item].set_index("timestamp").resample("W").mean(numeric_only=True)
    c = GROUP_COLORS[GROUP_OF[item]]
    fig.add_trace(go.Scatter(x=g.index, y=g["value"].clip(lower=1e-6), name=item, line=dict(color=c, width=1.5)), 1, 1)
    fig.add_trace(go.Scatter(x=g.index, y=g["y"], line=dict(color=c, width=1.5), showlegend=False), 1, 2)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_layout(template="plotly_white", height=420, title="Scaling · why we index before modelling (weekly means)")
fig.show()

In [ ]:
# 🗓️ Seasonal profiles and how strong each pattern is, per series
d = indexed(panel)
fig = make_subplots(rows=1, cols=3, subplot_titles=("Day of week", "Day of month", "Month of year"))
for j, (key, labels) in enumerate([(d.timestamp.dt.dayofweek, DOW), (d.timestamp.dt.day, None), (d.timestamp.dt.month, MON)], 1):
    prof = d.groupby(["group", key.rename("k")])["idx"].mean().reset_index()
    for gname, gg in prof.groupby("group"):
        fig.add_trace(go.Scatter(x=[labels[k] if labels else k for k in gg.k], y=gg.idx, name=gname, mode="lines+markers",
                                 line=dict(color=GROUP_COLORS[gname], width=2), showlegend=j == 1), 1, j)
fig.update_layout(template="plotly_white", height=400, title="Average profile by group (1.0 = the series' own mean)",
                  legend=dict(orientation="h", y=-0.15))
fig.show()

strength = seasonality_strength(panel)
order = meta.sort_values("group").item_id
fig = px.imshow(strength.loc[order].T, color_continuous_scale="Blues", zmin=0, zmax=1, aspect="auto",
                title="Seasonality strength · share of variance explained by each calendar cycle")
fig.update_layout(template="plotly_white", height=360, xaxis_tickangle=-60)
fig.show()

In [ ]:
# 🔥 The payment calendar in one picture: average flow by business day of the month
d = indexed(panel)
heat = d[d.bday_of_month <= 22].groupby(["item_id", "bday_of_month"])["idx"].mean().unstack()
heat = heat.loc[meta.item_id]
fig = px.imshow(heat, color_continuous_scale="Blues", zmin=0, zmax=4, aspect="auto",
                labels=dict(x="business day of the month", y="", color="× own mean"),
                title="Each line has its own payment rhythm (1.0 = an average day for that line)")
fig.update_layout(template="plotly_white", height=620)
fig.show()

In [ ]:
# 🔁 Autocorrelation (which lags carry signal) and a decomposition of one series
reps = meta.groupby("group").head(1).item_id.tolist()
max_lag = SEASON * 9
fig = make_subplots(rows=1, cols=len(reps), subplot_titles=reps, shared_yaxes=True)
for j, item in enumerate(reps, 1):
    a = acf(panel.loc[panel.item_id == item, "y"], max_lag)
    lags = np.arange(1, max_lag + 1)
    fig.add_trace(go.Bar(x=lags, y=a, marker_color=[ORANGE if l % SEASON == 0 else GREY for l in lags], showlegend=False), 1, j)
fig.update_layout(template="plotly_white", height=360,
                  title=f"Autocorrelation · orange bars are multiples of the weekly cycle ({SEASON} steps)")
fig.show()

dec = decompose(panel[panel.item_id == DECOMPOSE_ITEM]).iloc[-SEASON * 60:]
fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.03, subplot_titles=list(dec.columns))
for i, c in enumerate(dec.columns, 1):
    fig.add_trace(go.Scatter(x=dec.index, y=dec[c], line=dict(color=[DARK, BLUE, ORANGE, TEAL, GREY][i - 1], width=1.4),
                             showlegend=False), i, 1)
fig.update_layout(template="plotly_white", height=820, title=f"{DECOMPOSE_ITEM} · observed = trend × weekly × day-of-month × remainder")
fig.show()

In [ ]:
# 🎛️ Calendar covariates: catalogue, and the uplift each flag carries by group
catalogue = pd.DataFrame([{"feature": c, "family": "calendar (known in advance)", "description": COV_DOC.get(c, "")} for c in KNOWN]
                         + [{"feature": c, "family": "market driver (observed up to today)", "description": COV_DOC.get(c, "")} for c in PAST])
display(catalogue.style.set_caption("Covariates handed to the models").hide(axis="index"))

flags = [c for c in KNOWN if panel[c].dropna().isin([0, 1]).all()]
d = indexed(panel)
rows = {}
for c in flags:
    on = d[d[c] == 1].groupby("group")["idx"].mean()
    off = d[d[c] == 0].groupby("group")["idx"].mean()
    rows[c] = (on / off - 1) * 100
uplift = pd.DataFrame(rows).T
fig = px.imshow(uplift, color_continuous_scale="RdBu_r", zmin=-100, zmax=100, text_auto=".0f", aspect="auto",
                title="Average change in flow when a calendar flag is on (% vs days when it is off)")
fig.update_layout(template="plotly_white", height=40 * len(flags) + 160)
fig.show()

In [ ]:
# 🧪 Which engineered features matter? A gradient-boosting probe on a time-ordered hold-out
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error

feats = engineer_features(panel)
cut = feats.timestamp.quantile(0.85)
tr, te = feats[feats.timestamp <= cut], feats[feats.timestamp > cut]
cols = [c for c in feats.columns if c not in ("item_id", "timestamp", "y")]
fam = {c: feature_family(c) for c in cols}
print(f"{len(cols)} engineered features · {len(tr):,} training rows · {len(te):,} hold-out rows (after {cut:%d %b %Y})")
display(feats[["item_id", "timestamp", "y"] + cols[:8]].tail(3))

# Ablation: what each feature family adds
steps, used, maes = ["calendar", "lags", "rolling statistics", "market drivers"], [], {}
for s in steps:
    add = [c for c in cols if fam[c] == s]
    if not add:
        continue
    used += add
    m = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(tr[used], tr["y"])
    maes["+ " + s if len(maes) else s] = mean_absolute_error(te["y"], m.predict(te[used]))
naive = mean_absolute_error(te["y"], te[f"lag_{SEASON}"])

probe = te.sample(min(6000, len(te)), random_state=0)
imp = permutation_importance(m, probe[used], probe["y"], n_repeats=3, random_state=0, scoring="neg_mean_absolute_error")
imp = pd.Series(imp.importances_mean, index=used).sort_values().tail(18)

fig = make_subplots(rows=1, cols=2, column_widths=[0.4, 0.6],
                    subplot_titles=("Hold-out error as feature families are added", "Permutation importance (top 18)"))
fig.add_trace(go.Bar(x=["seasonal naive"] + list(maes), y=[naive] + list(maes.values()),
                     marker_color=[GREY] + [FAMILY_COLORS[s.replace("+ ", "")] for s in maes],
                     text=[f"{v:.3f}" for v in [naive] + list(maes.values())], textposition="outside", showlegend=False), 1, 1)
fig.add_trace(go.Bar(x=imp.values, y=imp.index, orientation="h", marker_color=[FAMILY_COLORS[fam[c]] for c in imp.index],
                     showlegend=False), 1, 2)
for f_, c_ in FAMILY_COLORS.items():
    fig.add_trace(go.Bar(x=[None], y=[None], marker_color=c_, name=f_), 1, 2)
fig.update_yaxes(title_text="MAE (indexed units)", row=1, col=1)
fig.update_layout(template="plotly_white", height=560, title="Feature engineering pays: lags, rolling statistics and calendar",
                  legend=dict(orientation="h", y=-0.1))
fig.show()

## 4 · Model race on rolling backtests

`TimeSeriesLeaderboard` scores every model on the last three 20-business-day windows of history, never on data it was fitted on. Foundation models run **zero-shot**: they have not been trained on Treasury data. **MASE** below 1 beats a seasonal-naive forecast; **WQL** scores the whole predictive distribution.

In [ ]:
# 🏁 Hold out the final horizon, race the models on rolling backtests of the rest
_past = {"past_covariates": PAST} if PAST else {}
schema = TimeSeriesSchema(target="y", item_id="item_id")
schema_cov = TimeSeriesSchema(target="y", item_id="item_id", known_covariates=KNOWN, **_past)
model_df = panel[["item_id", "timestamp", "y", *KNOWN, *PAST]]
history, actual = split_horizon(model_df, schema_cov, prediction_length=HORIZON)
future_cov = actual[["item_id", "timestamp", *KNOWN]]
history_y = history[["item_id", "timestamp", "y"]]
value_hist = panel.merge(history[["item_id", "timestamp"]], on=["item_id", "timestamp"])
value_act = panel.merge(actual[["item_id", "timestamp"]], on=["item_id", "timestamp"])
fp = {"prediction_length": HORIZON, "quantile_levels": QUANTILES}

board = TimeSeriesLeaderboard(history_y, schema, forecast_params=fp, windows=3)
board.add_models([m for m in BASELINES if m != "TabularTS-GBM"] + FOUNDATION_MODELS)
board.add_model("TabularTS-GBM", model_params=GBM_PARAMS)
t0 = time.time()
ranking = board.run(rank_by="mase", display=False)
print(f"{history.item_id.nunique()} series · {len(history):,} history rows · hold-out {HORIZON} steps · leaderboard in {time.time() - t0:.0f}s")
display(ranking)

ok = ranking[ranking["Status"].eq("ok")] if "Status" in ranking else ranking
fig = make_subplots(rows=1, cols=2, subplot_titles=("MASE (lower is better)", "WQL (lower is better)"))
for i, metric in enumerate(["mase", "wql"], start=1):
    o = ok.sort_values(metric)
    fig.add_trace(go.Bar(y=o["Model"], x=o[metric], orientation="h", text=o[metric].round(3), textposition="outside",
                         marker_color=[BLUE if m in FOUNDATION_MODELS else GREY for m in o["Model"]], showlegend=False), 1, i)
fig.add_vline(x=1.0, line_dash="dot", line_color=DARK, row=1, col=1)
fig.update_layout(template="plotly_white", height=420,
                  title="Rolling-backtest leaderboard · blue = foundation model (zero-shot), grey = baseline")
fig.show()

## 5 · Twenty-business-day forecast with calendar covariates

In [ ]:
# 🔮 Forecast with and without covariates, then calibrate the intervals
plain = TimeSeriesPipeline(PRIMARY, forecast_params=fp, license_mode="commercial", tuning_params={"device": DEVICE},
                           model_params=dict(PRIMARY_PARAMS))
plain.fit(history_y, schema)
m_plain = plain.evaluate(actual)

pipe = TimeSeriesPipeline(PRIMARY, forecast_params=fp, license_mode="commercial", tuning_params={"device": DEVICE},
                          model_params=dict(PRIMARY_PARAMS))
pipe.fit(history, schema_cov, future_df=future_cov)
m_cov = pipe.evaluate(actual, forecast=pipe.predict())

pipe.calibrate(windows=4)
forecast_cal = pipe.predict()
m_cal = pipe.evaluate(actual, forecast=forecast_cal)

KEYS = [k for k in ("mase", "wql", "smape", "coverage_80", "coverage_90") if k in m_cov]
display(pd.DataFrame({"target only": m_plain, "with covariates": m_cov, "with covariates, calibrated": m_cal}).loc[KEYS])
gain = (1 - m_cov["mase"] / m_plain["mase"]) * 100
display(Markdown(f"**Covariates change MASE by {gain:+.1f}%** on the hold-out ({PRIMARY})."))

fc = forecast_cal.to_pandas()
QCOLS = ["point"] + [str(q) for q in QUANTILES]
fc[QCOLS] = fc[QCOLS].mul(fc.item_id.map(SCALE), axis=0)          # back to business units

In [ ]:
# 🌀 Interactive fan chart: pick a series from the dropdown
tail = value_hist[value_hist.timestamp > value_hist.timestamp.max() - pd.Timedelta(days=150)]
items = list(meta.item_id)
fig = go.Figure()
for k, item in enumerate(items):
    h = tail[tail.item_id == item]; a = value_act[value_act.item_id == item]; f = fc[fc.item_id == item]
    vis = k == 0
    fig.add_trace(go.Scatter(x=h.timestamp, y=h.value, name="history", line=dict(color=DARK, width=1.5), visible=vis))
    fig.add_trace(go.Scatter(x=pd.concat([f.timestamp, f.timestamp[::-1]]), y=pd.concat([f["0.95"], f["0.05"][::-1]]),
                             fill="toself", fillcolor="rgba(47,91,234,0.12)", line=dict(width=0), name="90% band", visible=vis))
    fig.add_trace(go.Scatter(x=pd.concat([f.timestamp, f.timestamp[::-1]]), y=pd.concat([f["0.9"], f["0.1"][::-1]]),
                             fill="toself", fillcolor="rgba(47,91,234,0.25)", line=dict(width=0), name="80% band", visible=vis))
    fig.add_trace(go.Scatter(x=f.timestamp, y=f["0.5"], name="median forecast", line=dict(color=BLUE, width=2.5), visible=vis))
    fig.add_trace(go.Scatter(x=a.timestamp, y=a.value, name="actual", mode="markers", marker=dict(color=ORANGE, size=6), visible=vis))
n = 5
buttons = [dict(label=item, method="update",
                args=[{"visible": [j // n == k for j in range(n * len(items))]}, {"title": f"{item} · forecast"}])
           for k, item in enumerate(items)]
fig.update_layout(template="plotly_white", height=520, title=f"{items[0]} · forecast", yaxis_title=f"{UNIT} per day",
                  updatemenus=[dict(buttons=buttons, x=1.0, xanchor="right", y=1.15, direction="down")],
                  legend=dict(orientation="h", y=-0.15))
fig.show()

# Accuracy by series: where the model is strong and where it is not
per = fc.merge(value_act[["item_id", "timestamp", "value"]], on=["item_id", "timestamp"])
acc = per.groupby("item_id").apply(lambda g: pd.Series({
    "wape_%": (g["0.5"] - g["value"]).abs().sum() / max(g["value"].abs().sum(), 1e-9) * 100,
    "inside_90_%": g["value"].between(g["0.05"], g["0.95"]).mean() * 100})).reset_index()
acc["group"] = acc.item_id.map(GROUP_OF)
fig = px.bar(acc.sort_values("wape_%"), x="item_id", y="wape_%", color="group", color_discrete_map=GROUP_COLORS,
             title="Hold-out error by series (weighted absolute % error of the median forecast)")
fig.update_layout(template="plotly_white", height=430, xaxis_tickangle=-60, xaxis_title="")
fig.show()

## 6 · Treasury dashboard: net operating cash position

In [ ]:
# 🧭 Inflows minus outflows over the next 20 business days
sign = fc.item_id.map(lambda i: 1 if GROUP_OF[i] == "Inflow" else -1)
per = fc.merge(value_act[["item_id", "timestamp", "value"]], on=["item_id", "timestamp"])
per["sign"] = per.item_id.map(lambda i: 1 if GROUP_OF[i] == "Inflow" else -1)
net = per.assign(f=per["0.5"] * per.sign, a=per.value * per.sign).groupby("timestamp")[["f", "a"]].sum()
grp = per.assign(group=per.item_id.map(GROUP_OF)).groupby(["group", "timestamp"])[["0.5", "value"]].sum().reset_index()
tot = per.groupby("item_id").agg(forecast=("0.5", "sum"), actual=("value", "sum"))
tot["error_%"] = (tot.forecast / tot.actual.replace(0, np.nan) - 1) * 100
tot = tot.loc[meta.item_id]

fig = make_subplots(rows=2, cols=4, row_heights=[0.28, 0.72], vertical_spacing=0.13,
                    specs=[[{"type": "indicator"}] * 4, [{"colspan": 2}, None, {"colspan": 2}, None]],
                    subplot_titles=("", "", "", "", "Cumulative net operating cash flow (USD bn)", "20-day totals by line: forecast vs actual (USD bn)"))
inflow, outflow = tot.forecast[meta.group.values == "Inflow"].sum(), tot.forecast[meta.group.values == "Outflow"].sum()
kpis = [("Forecast inflow<br>20 days, USD bn", inflow / 1e3), ("Forecast outflow<br>20 days, USD bn", outflow / 1e3),
        ("Forecast net<br>USD bn", (inflow - outflow) / 1e3), ("Forecast error<br>hold-out MASE", m_cal["mase"])]
for i, (title, v) in enumerate(kpis, 1):
    fig.add_trace(go.Indicator(mode="number", value=v, title={"text": title},
                               number={"valueformat": ",.2f" if abs(v) < 50 else ",.0f", "font": {"size": 46}}), 1, i)
fig.add_trace(go.Scatter(x=net.index, y=net.f.cumsum() / 1e3, name="forecast (median)", line=dict(color=BLUE, width=3)), 2, 1)
fig.add_trace(go.Scatter(x=net.index, y=net.a.cumsum() / 1e3, name="actual", mode="lines+markers", line=dict(color=DARK, dash="dot")), 2, 1)
fig.add_trace(go.Bar(y=tot.index, x=tot.forecast / 1e3, orientation="h", name="forecast", marker_color=BLUE), 2, 3)
fig.add_trace(go.Bar(y=tot.index, x=tot.actual / 1e3, orientation="h", name="actual", marker_color=GREY), 2, 3)
fig.update_layout(template="plotly_white", height=820, barmode="group", legend=dict(orientation="h", y=-0.06),
                  title=f"Operating cash outlook · {fc.timestamp.min():%d %b} – {fc.timestamp.max():%d %b %Y} · model {PRIMARY}")
fig.show()
display(tot.style.format({"forecast": "{:,.0f}", "actual": "{:,.0f}", "error_%": "{:+.1f}%"})
        .background_gradient(subset=["error_%"], cmap="RdBu_r", vmin=-40, vmax=40)
        .set_caption("20-business-day totals per line (USD mn), hold-out period"))

## 7 · Ensembles and series similarity

In [ ]:
# 🤝 Greedy ensemble, and series similarity from foundation-model embeddings
members = [m if m != "TabularTS-GBM" else {"model_name": m, "model_params": GBM_PARAMS} for m in BASELINES + FOUNDATION_MODELS]
ensemble = TimeSeriesEnsemble(members, ensemble_strategy="greedy_selection",
                              forecast_params=fp, verbose=False)
ensemble.fit(history_y, schema)
display(ensemble.get_leaderboard())
m_ens = ensemble.evaluate(actual)
display(pd.Series({k: m_ens[k] for k in ("mase", "wql") if k in m_ens}, name="ensemble hold-out").to_frame())

if EMBEDDER:
    emb = TimeSeriesPipeline(EMBEDDER, task_type="embedding", tuning_params={"device": DEVICE}).fit(history_y, schema).predict()
    sim = pd.DataFrame(emb.similarity(), index=list(emb.item_ids), columns=list(emb.item_ids))
    order = meta.sort_values("group").item_id
    fig = px.imshow(sim.loc[order, order], color_continuous_scale="Blues", zmax=1,
                    title=f"Cosine similarity of {EMBEDDER} embeddings · {emb.dim}-dim per series")
    fig.update_layout(height=720, template="plotly_white"); fig.show()
else:
    print("Embeddings need a foundation model (skipped in fast mode).")

if RUN_FINETUNE and not FAST:
    tuned = TimeSeriesPipeline("ChronosBolt", tuning_strategy="peft", forecast_params=fp,
                               tuning_params={"epochs": 2, "steps_per_epoch": 50, "device": DEVICE}).fit(history_y, schema)
    zero = TimeSeriesPipeline("ChronosBolt", forecast_params=fp, tuning_params={"device": DEVICE}).fit(history_y, schema)
    display(pd.DataFrame({"zero-shot": zero.evaluate(actual), "LoRA, 2 epochs": tuned.evaluate(actual)}).loc[["mase", "wql"]])

## 8 · Anomaly replay: the COVID fiscal shock

Real data has no anomaly labels, so we replay history. The detector watches January 2019 to June 2021, with 2019 as its clean reference period, and each day is scored only from earlier data. A useful monitor should stay quiet through 2019 and light up from March 2020, when stimulus payments, unemployment benefits and deferred tax deadlines broke every seasonal pattern.

Because there are no labels, this section reports **where and when alerts fire**, not a detection score.

In [ ]:
def monitor_figure(res, feed_df, items, title_suffix, truth=None, events=None):
    """Expected band, observed flow, alerts (and true incidents or known events) with a series dropdown."""
    fig = go.Figure()
    per = 5
    for k, item in enumerate(items):
        r = res[res.item_id == item]; f = feed_df[feed_df.item_id == item]; vis = k == 0
        fig.add_trace(go.Scatter(x=pd.concat([r.timestamp, r.timestamp[::-1]]), y=pd.concat([r.upper, r.lower[::-1]]),
                                 fill="toself", fillcolor="rgba(47,91,234,0.14)", line=dict(width=0), name="expected range", visible=vis))
        fig.add_trace(go.Scatter(x=f.timestamp, y=f.y, name="observed", line=dict(color=DARK, width=1.3), visible=vis))
        if truth is not None:
            t = r.merge(truth[truth.is_anomaly], on=["item_id", "timestamp"])
            fig.add_trace(go.Scatter(x=t.timestamp, y=t.value, mode="markers", name="true incident", visible=vis,
                                     marker=dict(symbol="circle-open", size=14, color=ORANGE, line=dict(width=2))))
        else:
            fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="", showlegend=False, visible=vis))
        a = r[r.is_anomaly.fillna(False).astype(bool)]
        fig.add_trace(go.Scatter(x=a.timestamp, y=a.value, mode="markers", name="TabTune alert", visible=vis,
                                 marker=dict(symbol="x", size=9, color=RED)))
        fig.add_trace(go.Scatter(x=r.timestamp, y=r.score, name="anomaly score", yaxis="y2", line=dict(color=GREY, width=1), visible=vis))
    buttons = [dict(label=item, method="update",
                    args=[{"visible": [j // per == k for j in range(per * len(items))]}, {"title": f"{item} · {title_suffix}"}])
               for k, item in enumerate(items)]
    for when, label in (events or []):
        fig.add_vline(x=pd.Timestamp(when), line_dash="dot", line_color=GOLD)
        fig.add_annotation(x=pd.Timestamp(when), y=1.02, yref="paper", text=label, showarrow=False, textangle=-20, font=dict(size=10))
    fig.update_layout(template="plotly_white", height=580, title=f"{items[0]} · {title_suffix}",
                      yaxis=dict(title="indexed flow", domain=[0.3, 1]), yaxis2=dict(title="score", domain=[0, 0.22]),
                      updatemenus=[dict(buttons=buttons, x=1.0, xanchor="right", y=1.18)], legend=dict(orientation="h", y=-0.12))
    return fig

In [ ]:
# 🚨 Replay January 2019 – June 2021
EVENTS = [("2020-03-27", "CARES Act"), ("2020-04-15", "stimulus payments"), ("2020-07-15", "deferred tax deadline"),
          ("2021-01-04", "second stimulus"), ("2021-03-17", "third stimulus")]
replay = panel[(panel.timestamp >= "2019-01-01") & (panel.timestamp <= "2021-06-30")][["item_id", "timestamp", "y"]].reset_index(drop=True)
t0 = time.time()
det = TimeSeriesPipeline(DETECTOR, task_type="anomaly_detection", tuning_params={"device": DEVICE},
                         task_params={"method": "forecast_error", "min_context": 120, "alpha": 0.01, "reference_fraction": 0.25})
res = det.fit(replay, schema).predict().to_pandas()
print(f"scored {int(res.score.notna().sum()):,} series-days in {time.time() - t0:.0f}s · alerts: {int(res.is_anomaly.sum())}")

watch = ["Unemployment benefits", "Individual tax refunds", "Individual taxes, not withheld", "Other withdrawals", "Medicaid", "Federal tax deposits"]
monitor_figure(res, replay, watch + [i for i in meta.item_id if i not in watch], "anomaly replay", events=EVENTS).show()

In [ ]:
# 📋 Where and when the alerts fired
alerts = res[res.is_anomaly.fillna(False).astype(bool)].copy()
scored = res[res.score.notna()]
monthly = (alerts.groupby(["item_id", pd.Grouper(key="timestamp", freq="MS")]).size().unstack(fill_value=0)
           .reindex(index=meta.item_id, fill_value=0))
months = pd.date_range(scored.timestamp.min().to_period("M").to_timestamp(), scored.timestamp.max(), freq="MS")
monthly = monthly.reindex(columns=months, fill_value=0)
rate = (alerts.groupby(pd.Grouper(key="timestamp", freq="MS")).size().reindex(months, fill_value=0)
        / scored.groupby(pd.Grouper(key="timestamp", freq="MS")).size().reindex(months).clip(lower=1) * 100)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.05,
                    subplot_titles=("Alerts per line per month", "Share of series-days flagged (%)"))
fig.add_trace(go.Heatmap(z=monthly.values, x=monthly.columns, y=monthly.index, colorscale=[[0, "#F7F7F4"], [1, RED]], zmin=0,
                         colorbar=dict(title="alerts", len=0.6, y=0.7)), 1, 1)
fig.add_trace(go.Bar(x=rate.index, y=rate.values, marker_color=[RED if d >= pd.Timestamp("2020-03-01") else GREY for d in rate.index],
                     showlegend=False), 2, 1)
fig.add_hline(y=1.0, line_dash="dot", line_color=DARK, row=2, col=1)
for when, label in EVENTS:
    fig.add_vline(x=pd.Timestamp(when), line_dash="dot", line_color=GOLD)
fig.update_layout(template="plotly_white", height=760, title=f"Anomaly replay · {DETECTOR} · dotted line = the 1% false-alarm rate the detector was set to")
fig.show()

pre = scored.timestamp < "2020-03-01"
summary = pd.DataFrame({"series-days scored": [int(pre.sum()), int((~pre).sum())],
                        "alerts": [int((alerts.timestamp < "2020-03-01").sum()), int((alerts.timestamp >= "2020-03-01").sum())]},
                       index=["before March 2020", "March 2020 onwards"])
summary["alert rate %"] = summary.alerts / summary["series-days scored"] * 100
display(summary.style.format({"alert rate %": "{:.2f}%"}))
top = alerts.assign(deviation=(alerts.value - alerts.expected)).sort_values("score", ascending=False).head(12)
display(top[["timestamp", "item_id", "value", "expected", "score", "p_value"]].style.format({"value": "{:.2f}", "expected": "{:.2f}", "score": "{:.1f}", "p_value": "{:.4f}"})
        .set_caption("Strongest alerts (indexed units: 1.0 = the line's average day)"))

## Summary

| Stage | What was shown | TabTune call |
|---|---|---|
| Data preparation | 490k raw rows; structural rows removed; free-text labels from four reporting eras stitched into 18 continuous lines, with a continuity check; business-day grid and holidays | — |
| Feature engineering | scaling, seasonality strength, autocorrelation, decomposition, statutory payment and tax calendars, feature-family ablation | `TimeSeriesSchema(known_covariates=…)` |
| Model selection | rolling-backtest leaderboard, zero-shot foundation models vs baselines | `TimeSeriesLeaderboard` |
| Forecasting | calendar covariates, conformal calibration, fan charts | `TimeSeriesPipeline.fit / predict / calibrate` |
| Decision view | net operating cash position, per-line accuracy | — |
| Monitoring | unlabelled replay of 2019–2021 around the COVID shock | `task_type="anomaly_detection"` |

**Reading the results honestly**
* This is a public government ledger, used as a stand-in for a bank treasury's cash ledger; the pipeline, not the US budget, is the point.
* The label map in section 2b is explicit and auditable. Lines whose scope changed at a relabelling show up in the continuity table and should be read with that note.
* The anomaly replay has no ground truth; it shows alert timing against known events, not a detection rate.